<a href="https://colab.research.google.com/github/alearecuest/anyoneai-exercises-sprint_3/blob/main/17_7_1_RAG_Agent_with_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Agent with LangChain

This notebook demonstrates a simplified RAG (Retrieval Augmented Generation) system using **LangChain** with an agent-based approach.

## What's Inside

- **Document loading** from PDFs with chunking
- **Vector store** (FAISS) for efficient retrieval
- **ReAct Agent** that uses retrieval as a tool
- **Model**: Qwen/Qwen3-4B-Instruct-2507
- **Evaluation**: ROUGE-L, Context Precision, Recall, and Coverage metrics

## Key Benefits

- Clean LangChain abstractions simplify the code
- Agent can reason about when to retrieve information
- Easy to extend with additional tools

**Note**: As a demonstration, we're using a small model (only 4B parameters). That may cause a bad model performance in agentic tasks

## 1. Setup and Installation

In [ ]:
!pip install -q ragas==0.3.5 sentence-transformers transformers[torch] faiss-cpu pdfplumber langchain langchain-community langchain-huggingface langchain-openai rouge-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.3/284.3 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.8/473.8 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

## 2. Load Data
We download the same PDF used in the previous example.

In [ ]:
import os

# Create data directory
os.makedirs("/content/data", exist_ok=True)

# Download PDF
FILE_ID = "1EbDhFJlmxKtzcEZTurqw6KbNW9wJMMDR"
OUT_PATH = "/content/data/docs.pdf"

!gdown --id {FILE_ID} -O {OUT_PATH} || echo "gdown failed; make sure file id is correct"

print(f"File saved to {OUT_PATH}")

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1EbDhFJlmxKtzcEZTurqw6KbNW9wJMMDR
To: /content/data/docs.pdf
100% 1.05M/1.05M [00:00<00:00, 62.8MB/s]
File saved to /content/data/docs.pdf


## 3. Document Loading and Splitting
We use `PDFPlumberLoader` to load the PDF and `RecursiveCharacterTextSplitter` to chunk it.

In [ ]:
from langchain_community.document_loaders import PDFPlumberLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Load
loader = PDFPlumberLoader(OUT_PATH)
docs = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)
splits = text_splitter.split_documents(docs)

print(f"Loaded {len(docs)} pages.")
print(f"Created {len(splits)} chunks.")

Loaded 12 pages.
Created 180 chunks.


## 4. Vector Store (FAISS)
We use `HuggingFaceEmbeddings` (all-mpnet-base-v2) and store vectors in FAISS.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Embeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")

# Vector Store
vectorstore = FAISS.from_documents(documents=splits, embedding=embedding_model)

# Create a retriever interface
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("Vector store created and retriever ready.")

Vector store created and retriever ready.


## 5. Setup LLM (Qwen 3 4B Instruct)
We load the model locally using HuggingFacePipeline.

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

model_id = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Create pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False, # Deterministic for RAG
    return_full_text=False,
    temperature=0.1
)

llm = HuggingFacePipeline(pipeline=pipe)



# TO USE AN OPEN AI MODEL:
#from langchain_openai import ChatOpenAI
#import os

# Make sure your API key is set
#os.environ["OPENAI_API_KEY"] = "sk-"

#model_id = "gpt-4o-mini"

#llm = ChatOpenAI(
#    model=model_id,
#    temperature=0.1,
#)

print("LLM initialized.")


LLM initialized.


## 6. Create Retrieval Tool and Agent
We wrap the retriever in a Tool and create a ReAct agent.

In [ ]:
from langchain.agents import initialize_agent, AgentType
from langchain_core.tools import Tool

# Define the retrieval tool
def retrieve_func(query):
    docs = retriever.invoke(query)
    return "\n\n".join([
        f"Content: {d.page_content}\nSource: {d.metadata.get('source', 'unknown')}"
        for d in docs
    ])

tools = [
    Tool(
        name="TRIPOD_Knowledge_Base",
        func=retrieve_func,
        description=(
            "Use this tool to retrieve information about TRIPOD-LLM, "
            "reporting guidelines, and healthcare AI studies. "
            "Input should be a natural-language question."
        )
    )
]

agent_executor = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=4
)

print("Agent ready.")

Agent ready.


## 7. Run the Agent
Now we can ask questions.

In [ ]:
# Example Question 1
query = "What is TRIPOD-LLM?"
response = agent_executor.invoke({"input": query})
print("\nFinal Answer:", response["output"])



> Entering new AgentExecutor chain...
I need to find information about TRIPOD-LLM to answer the question. 
Action: TRIPOD_Knowledge_Base
Action Input: What is TRIPOD-LLM?
Observation: Content: lected from the community via multiple avenues to enhance accessi-
bility — a project-specific GitHub repository, the TRIPOD-LLM website 1. No modification.
and the main TRIPOD website (https://www.tripod-statement.org/). 2. Modification of substantive content (small, editorial revisions
We encourage input both on usability, such as language that may be such as rewording for clarity and and correcting types will not
Source: /content/data/docs.pdf

Content: of the guidelines at https://tripod-llm.vercel.app/. review and subject matter expertise. The steering committee
Our approach to the living TRIPOD-LLM statement is informed by will revise the statement based on this discussion, and this will be
processes established in developing living systematic reviews33,34 and circulated to the expert pan

In [ ]:
# Example Question 2
query = "How many items are in the TRIPOD-LLM checklist?"
response = agent_executor.invoke({"input": query})
print("\nFinal Answer:", response["output"])



> Entering new AgentExecutor chain...
To answer the question about the number of items in the TRIPOD-LLM checklist, I need to retrieve specific information regarding the TRIPOD-LLM guidelines. 
Action: TRIPOD_Knowledge_Base
Action Input: "How many items are in the TRIPOD-LLM checklist?"
Observation: Content: How it works Example scenario
The TRIPOD-LLM checklist starts
The complete TRIPOD-LLM checklist Total checklist items with 59 items for reporting,
contains a list of all items that should be
covering background and related
reported when publishing LLM studies;
h ev o e w r e y v s e t r u , d n y o . t all items are relevant to 59 w re o p r r k o , d m u e c t ib h i o li d ty s a a n n d d f s u ta n t d is in ti g c , s,
findings and sensitivity.
After selecting research task(s)
Source: /content/data/docs.pdf

Content: the field evolves, task(s), design(s) dynamically display a filtered list
and items may be updated to reflect of checklist items for you to
new knowledge in the

## 8. Evaluation Samples
We'll evaluate the RAG system using the same samples as the original notebook.

In [ ]:
eval_samples = [
    {
        "query": "What is TRIPOD-LLM?",
        "reference": "An extension of TRIPOD+AI providing a checklist for transparent reporting of studies using large language models in healthcare."
    },
    {
        "query": "How many main items and subitems are in the TRIPOD-LLM checklist?",
        "reference": "TRIPOD-LLM provides 19 main items and 50 subitems."
    },
    {
        "query": "What is the purpose of the TRIPOD-LLM interactive website?",
        "reference": "To present required questions based on research design(s) and task(s), allow easy completion and render a final PDF suitable for submission."
    },
    {
        "query": "Name three noteworthy changes introduced in TRIPOD-LLM (Box 1).",
        "reference": "A new dedicated checklist for LLMs; a living guideline (regular updates); and task-specific guidance tailored to LLM applications in healthcare."
    },
    {
        "query": "Which key task categories are listed for TRIPOD-LLM (examples)?",
        "reference": "Examples include long-form question answering, information retrieval, conversational agents (chatbots), documentation generation, summarization and simplification, machine translation and outcome forecasting."
    },
    {
        "query": "What does item 5a of the checklist ask authors to report?",
        "reference": "Describe the sources of data separately for training, tuning and/or evaluation datasets and the rationale for using these data."
    },
    {
        "query": "What metrics does item 7a recommend including for generative outputs?",
        "reference": "Metrics that capture consistency, relevance, accuracy and presence/type of errors compared to gold standards."
    },
    {
        "query": "Who are the primary intended users and beneficiaries of TRIPOD-LLM?",
        "reference": "Academic and industry researchers, journal editors and peer reviewers, and other stakeholders such as policymakers, funders, regulators, patients and study participants."
    },
    {
        "query": "What open-science items should be reported (brief)?",
        "reference": "Report source of funding and role of funders; conflicts of interest; protocol access; registration information; availability of study data and code."
    },
    {
        "query": "How should studies that include imaging (multimodal) report aspects differently?",
        "reference": "For vision-language or multimodal models, both text and image preprocessing and image acquisition details should be reported, with additional considerations possible in future versions."
    }
]

print(f"Prepared {len(eval_samples)} evaluation samples.")

Prepared 10 evaluation samples.


## 9. Run RAG over Evaluation Samples

In [ ]:
generated_results = []

for s in eval_samples:
    query = s["query"]
    # Get retrieved context
    retrieved_docs = retriever.invoke(query)
    retrieved = [{"text": doc.page_content, "source": doc.metadata.get("source", "unknown")} for doc in retrieved_docs]

    # Generate answer using agent (with limited verbosity for cleaner output)
    agent_executor.verbose = False
    try:
        response = agent_executor.invoke({"input": query})
        answer = response["output"]
    except Exception as e:
        answer = str(e)

    generated_results.append({
        "query": query,
        "generated": answer,
        "reference": s["reference"],
        "retrieved": retrieved
    })

# Restore verbose mode
agent_executor.verbose = True

# Display results
for gr in generated_results:
    print("Q:", gr["query"])
    print("Gen:", gr["generated"][:300])
    print("Ref:", gr["reference"][:300])
    print("-"*60)

Q: What is TRIPOD-LLM?
Gen: TRIPOD-LLM is a reporting guideline specifically designed for studies involving large language models (LLMs). It aims to enhance the transparency and quality of research by providing a checklist of items that should be reported when publishing LLM studies. The checklist includes 19 main items coveri
Ref: An extension of TRIPOD+AI providing a checklist for transparent reporting of studies using large language models in healthcare.
------------------------------------------------------------
Q: How many main items and subitems are in the TRIPOD-LLM checklist?
Gen: Agent stopped due to iteration limit or time limit.
Ref: TRIPOD-LLM provides 19 main items and 50 subitems.
------------------------------------------------------------
Q: What is the purpose of the TRIPOD-LLM interactive website?
Gen: The purpose of the TRIPOD-LLM interactive website is to enhance understanding and appraisal of LLM study methods in healthcare, increase transparency around study find

## 10. Understanding the Evaluation Metrics

This section explains the meaning of each metric computed in the evaluation step.

---

### **1. ROUGE-L**
**What it measures:**  
The overlap between the *generated answer* and the *reference (ground truth)* answer, based on the *longest common subsequence (LCS)* of words.

**Interpretation:**  
- High ROUGE-L → The generated answer textually matches the correct answer well.  
- Low ROUGE-L → The generated answer diverges from the reference (either incomplete or phrased differently).

**Use:**  
A general proxy for answer quality and textual similarity.

---

### **2. Context Precision**
**Definition:**  
The proportion of tokens in the *generated answer* that also appear in the *retrieved context passages*.

**Formula:**  
$$
\text{Context Precision} = \frac{\text{# tokens in generated answer also in retrieved text}}{\text{total # tokens in generated answer}}
$$

**Interpretation:**  
- High → The model primarily used content that appears in the retrieved passages (indicating grounded, faithful answers).  
- Low → The model introduced content not supported by the retrieved context (possible hallucination).

---

### **3. Context Recall**
**Definition:**  
The proportion of tokens in the *reference (ground truth)* answer that are present in the *retrieved context*.

**Formula:**  
$$
\text{Context Recall} = \frac{\text{# tokens in reference answer also in retrieved text}}{\text{total # tokens in reference answer}}
$$

**Interpretation:**  
- High → The retrieved passages contain most of the information needed for the correct answer (good retrieval coverage).  
- Low → Important information from the reference was missing in the retrieved content (poor retrieval).

---

### **4. Context Coverage**
**Definition:**  
In this simplified framework, *coverage* is equivalent to **context recall** and is included to highlight how much of the true answer is represented in the retrieved material.

---

### **Summary of Interpretation**

| Metric | Measures | High Value Means | Low Value Means |
|---------|-----------|------------------|------------------|
| **ROUGE-L** | Overlap between generated and reference answers | Good textual alignment | Divergent or incomplete answer |
| **Context Precision** | Grounding of generated text in retrieved context | Faithful, low hallucination | Unsupported or hallucinated text |
| **Context Recall** | Retrieval coverage of correct information | Relevant passages retrieved | Missing key information |
| **Context Coverage** | Portion of true answer covered by retrieval | Good evidence coverage | Incomplete retrieval |

---

### **Typical Ranges (Rule of Thumb)**
- **ROUGE-L > 0.5** → Acceptable content similarity.  
- **Precision > 0.7** → Generation mostly grounded in retrieved text.  
- **Recall > 0.5** → Retrieval covered most relevant content.

These thresholds are heuristic. They are most useful for **comparing models or retrieval strategies** within the same experimental setup.

In [ ]:
# Simple deterministic RAG evaluation (ROUGE-L + context overlap metrics)
import json, re, statistics
from rouge_score import rouge_scorer
from pprint import pprint

# Helper: simple tokeniser (word characters)
def tokenize(text):
    if not text:
        return []
    return re.findall(r"\w+", text.lower())

# Compute ROUGE-L using rouge_score library
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

per_sample = []
for entry in generated_results:
    query = entry.get("query", "")
    generated = (entry.get("generated") or "").strip()
    reference = (entry.get("reference") or "").strip()

    # ROUGE-L score
    rouge_scores = scorer.score(reference, generated)
    rouge_l_f = rouge_scores['rougeL'].fmeasure  # F1-like score for ROUGE-L

    # Build a single string of all retrieved context text
    retrieved_texts = [r.get("text","") for r in entry.get("retrieved", [])]
    combined_context = " ".join(retrieved_texts)

    # Token counts
    gen_tokens = tokenize(generated)
    ref_tokens = tokenize(reference)
    ctx_tokens = set(tokenize(combined_context))

    # Context precision: fraction of generated tokens also present in contexts
    ctx_prec = 0.0
    if gen_tokens:
        matched_gen = sum(1 for t in gen_tokens if t in ctx_tokens)
        ctx_prec = matched_gen / len(gen_tokens)

    # Context recall: fraction of reference tokens present in contexts
    ctx_rec = 0.0
    if ref_tokens:
        matched_ref = sum(1 for t in ref_tokens if t in ctx_tokens)
        ctx_rec = matched_ref / len(ref_tokens)

    # Context coverage is same as ctx_rec but provided separately for clarity
    ctx_cov = ctx_rec

    per_sample.append({
        "query": query,
        "generated": generated,
        "reference": reference,
        "rouge_l_f": rouge_l_f,
        "context_precision": ctx_prec,
        "context_recall": ctx_rec,
        "context_coverage": ctx_cov
    })

# Aggregate statistics (means and standard deviations)
def mean_std(values):
    if not values:
        return {"mean": None, "std": None}
    return {"mean": statistics.mean(values), "std": statistics.pstdev(values)}

rouge_vals = [s["rouge_l_f"] for s in per_sample]
prec_vals = [s["context_precision"] for s in per_sample]
rec_vals = [s["context_recall"] for s in per_sample]
cov_vals = [s["context_coverage"] for s in per_sample]

summary = {
    "n_samples": len(per_sample),
    "rouge_l": mean_std(rouge_vals),
    "context_precision": mean_std(prec_vals),
    "context_recall": mean_std(rec_vals),
    "context_coverage": mean_std(cov_vals)
}

results = {"summary": summary, "per_sample": per_sample}

# Save results
out_path = "/content/langchain_rag_eval.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)

print("Saved LangChain RAG evaluation to", out_path)
print("\nSummary:")
pprint(summary)
print("\nPer-sample preview (first 3):")
pprint(per_sample[:3])

Saved LangChain RAG evaluation to /content/langchain_rag_eval.json

Summary:
{'context_coverage': {'mean': 0.555120409719952, 'std': 0.23078304676963693},
 'context_precision': {'mean': 0.5517363845305022, 'std': 0.33113642265332555},
 'context_recall': {'mean': 0.555120409719952, 'std': 0.23078304676963693},
 'n_samples': 10,
 'rouge_l': {'mean': 0.17988635761716965, 'std': 0.21272140727983274}}

Per-sample preview (first 3):
[{'context_coverage': 0.5789473684210527,
  'context_precision': 0.7032967032967034,
  'context_recall': 0.5789473684210527,
  'generated': 'TRIPOD-LLM is a reporting guideline specifically designed for '
               'studies involving large language models (LLMs). It aims to '
               'enhance the transparency and quality of research by providing '
               'a checklist of items that should be reported when publishing '
               'LLM studies. The checklist includes 19 main items covering '
               'aspects such as the title, abstract